# Importeren van libraries

In [1]:
import pandas as pd
#Pandas om dataframes te maken en analyseren
import pyarrow
from pathlib import Path
#Path om binnen de omgeving .csv-files te zoeken en laden.
import re
#Regex module importeren om pattern matching te gebruiken.
from shared import init_analyse
#Uit shared py de init_analyse functie. 
#Een kleine functie op basis van de eerste analyse waarbij dezelfde functies binnen pandas op 1 dataframe worden gebruikt.
pd.options.display.float_format = "{:,.2f}".format
#Floats weergeven als getallen met 2 decimalen achter de komma, aanvankelijk neigt pandas bij grotere getallen naar wetenschappelijke weergaven.

# 1 ***Analyse sales data***

In dit onderdeel importeren en analyseren we de salesdata met behulp van pandas

## 1.1 Importeren en inlezen dataset

In [2]:
sales_path= Path(r"Data\Contoso_Sales.csv")
#Inlezen sales.csv
sales_df = pd.read_csv(sales_path,
                       decimal=",")
#Dataframe koppelen aan variabele
#Decimal argument meegegeven, omdat pandas standaard "." als delimiter ziet.

Data is met behulp van pathlib en pandas ingelezen.  
Bij het inlezen ontstond initieel een dtypewarning voor de kolommen ReturnAmount en DiscountAmount. 
Door bij het inlezen het decimal teken mee te geven wordt deze fout verholpen en worden floats gelijk correct ingelezen.

## 1.2. Eerste analyse
Hier wordt de algehele dataset geanalyseerd op afwijkingen, waar hierna op groter detailniveau naar gekeken wordt.

In [3]:
sales_df.info()
#Eerste data over kolommen opvragen

<class 'pandas.DataFrame'>
RangeIndex: 2282482 entries, 0 to 2282481
Data columns (total 15 columns):
 #   Column            Dtype  
---  ------            -----  
 0   SalesKey          int64  
 1   DateKey           str    
 2   channelKey        int64  
 3   StoreKey          int64  
 4   ProductKey        int64  
 5   PromotionKey      int64  
 6   UnitCost          float64
 7   UnitPrice         float64
 8   SalesQuantity     int64  
 9   ReturnQuantity    int64  
 10  ReturnAmount      float64
 11  DiscountQuantity  int64  
 12  DiscountAmount    float64
 13  TotalCost         float64
 14  SalesAmount       float64
dtypes: float64(6), int64(8), str(1)
memory usage: 311.3 MB


- Datekey wordt als string weergegeven, deze dient nog omgezet te worden naar een tijdreeks.
- UnitCost, UnitPrice, TotalCost en SalesAmount werden initieel als strings geïnterpreteerd. Dit is afgevangen door bij de import de parameter decimal mee te geven en komma's aan te merken als het decimaal-teken.

In [4]:
sales_df.head(20)
#De eerste 20 rijn tonen met waarden.

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
0,2201,"2011-05-07 00:00:00,000",1,76,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
1,26527,"2013-05-22 00:00:00,000",1,26,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
2,26702,"2013-09-30 00:00:00,000",1,298,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
3,33125,"2012-10-31 00:00:00,000",1,264,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
4,52306,"2013-06-26 00:00:00,000",1,98,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
5,67230,"2012-10-28 00:00:00,000",1,194,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
6,79246,"2012-05-12 00:00:00,000",1,230,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
7,87210,"2013-04-17 00:00:00,000",1,84,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
8,91129,"2013-05-26 00:00:00,000",1,215,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
9,109148,"2013-04-06 00:00:00,000",1,93,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"


Datekey lijkt een datumwaarde te moeten zijn. Op het moment is dit een string. Deze dient bij het opschonen omgezet te worden, zodat deze als key gebruikt kan worden.  
UnitCost, UnitPrice, TotalCost en SalesAmount lijkt om bedragen te gaan. De stuksprijs maal de quantity is hier ook gelijk aan de Totalcost en SalesAmount.  
Gezien het feit er geen strings in de tabel zitten, is er geen noodzaak om te controleren of deze afwijken van elkaar in schrijfwijze. Hiermee wordt bedoeld waarden die dezelfde categorie betreffen, maar andere schrijfwijze hebben door gebruik van hoofdletters en leestekens. 

In [5]:
sales_df.tail(20)
#Laatste 20 waarden bekijken.

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
2282462,3366917,"2011-06-28 00:00:00,000",1,120,613,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282463,3367028,"2011-05-20 00:00:00,000",1,12,320,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282464,3369192,"2012-05-15 00:00:00,000",1,60,320,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282465,3369889,"2013-04-29 00:00:00,000",1,167,201,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282466,3369908,"2012-10-06 00:00:00,000",1,115,405,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282467,3370934,"2012-04-21 00:00:00,000",1,142,624,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282468,3371111,"2013-05-23 00:00:00,000",1,21,345,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282469,3372220,"2011-05-29 00:00:00,000",1,2,613,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282470,3378122,"2011-06-25 00:00:00,000",1,243,405,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282471,3378463,"2013-10-26 00:00:00,000",1,282,345,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"


Laatste 20 rijen laten hetzelfde patroon zien. In deze rijen zitten ook returns verwerkt. Deze lijken wel goed verwerkt te worden in de TotalCost, bijvoorbeeld 10 items verkocht - 1 return = 9 * UnitCost.  
Op salesniveau lijkt dit echter niet correct doorberekend. De SalesAmount blijft gelijk aan 10 maal de UnitPrice, ondanks de return.  
Huidige actie lijkt  om een extra kolom toe te voegen die de SalesAmount berekend met inachtname van Returns. Dit houdt de originele kolom in tact, maar geeft een betere representatie van de daadwerkelijke omzet van een order.

In [6]:
sales_df.describe()

,SalesKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
count,"2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00"
mean,"1,703,424.16",1.63,196.38,881.19,8.50,124.60,297.18,16.17,0.14,42.56,1.48,55.52,"1,556.38","3,654.45"
std,"983,152.96",1.03,95.54,509.48,8.26,126.75,347.31,39.35,0.36,172.02,1.80,139.05,"1,888.55","4,853.79"
min,1.00,1.00,1.00,1.00,1.00,0.48,0.95,4.00,0.00,0.00,0.00,0.00,1.92,3.04
25%,"851,809.50",1.00,121.00,453.00,1.00,30.58,66.00,9.00,0.00,0.00,0.00,0.00,321.48,658.42
50%,"1,704,423.50",1.00,200.00,880.00,4.00,87.37,200.00,10.00,0.00,0.00,1.00,2.42,986.04,"2,293.60"
75%,"2,554,564.75",2.00,296.00,"1,273.00",14.00,162.64,358.00,13.00,0.00,0.00,3.00,53.98,"2,065.23","4,699.70"
max,"3,406,089.00",4.00,310.00,"2,517.00",28.00,960.82,"2,899.99","2,880.00",5.00,"9,996.00",24.00,"6,497.40","84,115.20","235,467.00"


In [7]:
sales_df.isna().sum()
#Controleren op lege waarden in cellen. Dit is niet van toepassing.


SalesKey            0
DateKey             0
channelKey          0
StoreKey            0
ProductKey          0
PromotionKey        0
UnitCost            0
UnitPrice           0
SalesQuantity       0
ReturnQuantity      0
ReturnAmount        0
DiscountQuantity    0
DiscountAmount      0
TotalCost           0
SalesAmount         0
dtype: int64

In [8]:
sales_df.nunique() 
#Aantal unique waarden opsommen.

SalesKey            2282482
DateKey                1096
channelKey                4
StoreKey                306
ProductKey             1689
PromotionKey             28
UnitCost                397
UnitPrice               350
SalesQuantity            98
ReturnQuantity            6
ReturnAmount            782
DiscountQuantity         23
DiscountAmount         8572
TotalCost             14114
SalesAmount           51717
dtype: int64

In [9]:
sales_df[sales_df.duplicated()]
#Checken op duplicated rows

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount


Op basis van bovenstaande analyses zijn er geen lege waardes of duplicaten aanwezig in deze dataset.

De volgende keys dienen gecontroleerd te worden op aanwezigheid in de andere dataset:
- 4 verkoopkanalen volgens sales set.
- 306 unieke winkels volgens sales set.
- 1689 soort producten
- 28 promoties
- 783 retourregels, hoewel er geen verdere datasets met betrekking tot retourproducten lijkt te zijn.

## 1.3 Clean up

Op basis van de analyse zien we dat de datum gegevens nu niet als juiste dtype worden ingegeven.  
In dit onderdeel worden deze omgezet naar bruikbaar datums.

In [10]:
sales_df2 = sales_df.copy()
#Dataframe kopieeëren als backup. Zo blijft de originele dataset altijd behouden ter controle. 
sales_df2

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
0,2201,"2011-05-07 00:00:00,000",1,76,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
1,26527,"2013-05-22 00:00:00,000",1,26,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
2,26702,"2013-09-30 00:00:00,000",1,298,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
3,33125,"2012-10-31 00:00:00,000",1,264,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
4,52306,"2013-06-26 00:00:00,000",1,98,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282477,3392414,"2013-05-15 00:00:00,000",1,40,397,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282478,3395082,"2013-10-04 00:00:00,000",1,52,565,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282479,3396997,"2011-07-26 00:00:00,000",1,303,381,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282480,3398163,"2013-07-04 00:00:00,000",1,247,201,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"


In [11]:
sales_df2["DateKey"] = pd.to_datetime(sales_df2["DateKey"], format="%Y-%m-%d %H:%M:%S,%f")
#Omzetten naar datetime van DateKey. De originele string bevat gegevens tot en met microseconden dus dient hier rekening mee gehouden worden bij de conversie.
display(sales_df2)

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
0,2201,2011-05-07,1,76,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
1,26527,2013-05-22,1,26,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
2,26702,2013-09-30,1,298,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
3,33125,2012-10-31,1,264,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
4,52306,2013-06-26,1,98,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282477,3392414,2013-05-15,1,40,397,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282478,3395082,2013-10-04,1,52,565,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282479,3396997,2011-07-26,1,303,381,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282480,3398163,2013-07-04,1,247,201,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"


In [12]:
print(sales_df2.loc[sales_df2["DateKey"].idxmin(),"DateKey"])
print(sales_df2.loc[sales_df2["DateKey"].idxmax(),"DateKey"])
#Kijken naar begin en einddatum van dataset.

2011-01-01 00:00:00
2013-12-31 00:00:00


De dataset loopt van 1 januari 2011 tot en met 31 december 2013

In [13]:
datesales = sales_df2.set_index("DateKey")
#Nieuwe sales dataframe met date als index, zodat resamplen mogelijk is.
datesales.sort_values(by="DateKey")
#Sorteren van nieuwe naar oud.

,SalesKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
DateKey,,,,,,,,,,,,,,
2011-01-01,2725067,1,99,1268,2,25.47,49.96,4,0,0.00,4,9.99,101.88,189.85
2011-01-01,1490686,1,288,1634,2,5.09,9.99,4,0,0.00,3,1.50,20.36,38.46
2011-01-01,28547,1,54,446,2,112.14,219.95,12,0,0.00,4,43.99,"1,345.68","2,595.41"
2011-01-01,1022476,1,15,1142,2,180.22,391.90,9,0,0.00,0,0.00,"1,621.98","3,527.10"
2011-01-01,359421,1,103,1008,2,84.84,184.50,9,0,0.00,4,36.90,763.56,"1,623.60"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-31,3170307,4,310,1081,23,188.19,568.00,13,0,0.00,3,255.60,"2,446.47","7,128.40"
2013-12-31,2692979,1,255,1342,23,13.92,42.00,6,0,0.00,2,12.60,83.52,239.40
2013-12-31,1227951,1,238,1367,28,16.55,35.99,8,0,0.00,2,14.40,132.40,273.52


In [14]:
SalesByMonth = datesales.resample("ME").agg(TotalKeys=("SalesKey","nunique"),
                             VerkoopKanalen=("channelKey","nunique"),
                             TotalCost=("TotalCost","sum"),
                             SalesAmount=("SalesAmount","sum")
)
#Dataframe die kijkt naar aantal verkopen, kanalen, totale kosten en totale omzet per maand.

In [15]:
SalesByMonth["Brutomarge"] = SalesByMonth["SalesAmount"] - SalesByMonth["TotalCost"]
#Kolom Brutomarge toevoegen aan dataframe.
SalesByMonth["BrutomargePct"] = SalesByMonth["Brutomarge"]/SalesByMonth["SalesAmount"]
#Brutomarge uitdrukken als percentage van de totale omzet.
display(SalesByMonth)

,TotalKeys,VerkoopKanalen,TotalCost,SalesAmount,Brutomarge,BrutomargePct
DateKey,,,,,,
2011-01-31,73868,4,"81,593,589.36","193,305,554.64","111,711,965.28",0.58
2011-02-28,70212,4,"88,412,165.20","209,439,067.93","121,026,902.73",0.58
2011-03-31,73596,4,"86,905,506.38","203,991,979.69","117,086,473.31",0.57
2011-04-30,91338,4,"116,872,052.05","276,891,048.16","160,018,996.11",0.58
2011-05-31,94223,4,"122,236,876.68","288,749,508.61","166,512,631.93",0.58
2011-06-30,92553,4,"119,806,180.24","283,186,644.54","163,380,464.30",0.58
2011-07-31,76193,4,"115,362,984.04","272,818,635.11","157,455,651.07",0.58
2011-08-31,76085,4,"111,509,275.12","263,780,279.28","152,271,004.16",0.58
2011-09-30,74802,4,"108,858,863.84","257,282,781.95","148,423,918.11",0.58


### **Tussentijdse bevindingen**

De sales dataset loopt van 1 januari 2011 tot en met 31 december 2013 en bestaat uit ***2.282.482*** unieke saleskeys.
Het is nog onduidelijk of de return amounts verrekend moeten worden met de SalesAmount, gezien het verschil tussen kosten en opbrengsten bij een retour.  
Het overzicht op maandniveau laat zien dat de brutomarge redelijk constant is over de gegeven periode. Het aantal verkopen lijkt wel af te nemen.
De datetimes zijn reeds omgezet.

De volgende keys dienen gecontroleerd te worden op aanwezigheid in de andere dataset:
- 4 **verkoopkanalen** volgens sales set.
- 306 **unieke winkels** volgens sales set.
- 1689 soort **producten**
- 28 **promoties**
- 783 **retourregels**

*To-do*:  
- Aanmaken extra kolom waarin SalesAmount rekening houdt met ReturnAmount. 

In [16]:
sales_df2["SalesAmountRet"] = (sales_df2["SalesQuantity"]*sales_df2["UnitPrice"])-(sales_df2["ReturnQuantity"]*sales_df2["UnitPrice"])
#Aparte kolom maken waarin returns worden verrekend met de omzet.

In [17]:
sales_df2.to_parquet(r"PQ/Contoso_sales.parquet")

In [18]:
pq = pd.read_parquet(r"PQ/Contoso_sales.parquet")
#Testen of de pq file gelezen kan worden.

# 2 ***Analyse product data***
In dit onderdeel wordt de product data set verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data set. Naar verwachting zijn er minimaal 1689 producten in de dataset.

## 2.1 Importeren dataset
De dataset wordt geimporteerd met behulp van pandas en path.

In [19]:
product_path= Path(r"Data\Contoso_Product.csv")
#Inlezen sales.csv
product_df = pd.read_csv(product_path,
                       decimal=",")

## 2.2 Analyseren product dataset

In [20]:
init_analyse(product_df)
#Functie aanhalen uit shared.py geschreven om te voorkoemn dat telkens dezelfde stappen doorlopen hoeven te worden.

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 1690 entries, 0 to 1689
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   BrandName              1690 non-null   str  
 1   ClassName              1690 non-null   str  
 2   Manufacturer           1690 non-null   str  
 3   ProductDescription     1690 non-null   str  
 4   ProductKey             1690 non-null   int64
 5   ProductName            1690 non-null   str  
 6   ProductSubcategoryKey  1690 non-null   int64
 7   UnitCost               1690 non-null   str  
 8   UnitPrice              1690 non-null   str  
dtypes: int64(2), str(7)
memory usage: 374.9 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


BrandName                0
ClassName                0
Manufacturer             0
ProductDescription       0
ProductKey               0
ProductName              0
ProductSubcategoryKey    0
UnitCost                 0
UnitPrice                0
dtype: int64


Aantal unieke waarden:


BrandName                  10
ClassName                   3
Manufacturer               10
ProductDescription        516
ProductKey               1690
ProductName              1690
ProductSubcategoryKey      22
UnitCost                  397
UnitPrice                 350
dtype: int64


Eerste 20 rijen:


,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
0,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",951,A. Datum Consumer Digital Camera E100 Black,23,"$143,26",$281
1,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",965,A. Datum Consumer Digital Camera E100 Grey,23,"$143,26",$281
2,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",979,A. Datum Consumer Digital Camera E100 Pink,23,"$143,26",$281
3,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",993,A. Datum Consumer Digital Camera E100 Silver,23,"$143,26",$281
4,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",1007,A. Datum Consumer Digital Camera E100 Orange,23,"$143,26",$281
5,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",1021,A. Datum Consumer Digital Camera E100 Green,23,"$143,26",$281
6,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",1035,A. Datum Consumer Digital Camera E100 Azure,23,"$143,26",$281
7,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, Product type digital camera - ...",950,A. Datum Super-zoom Digital Camera X300 Black,23,"$96,08",$290
8,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, Product type digital camera - ...",964,A. Datum Super-zoom Digital Camera X300 Grey,23,"$96,08",$290
9,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, Product type digital camera - ...",978,A. Datum Super-zoom Digital Camera X300 Pink,23,"$96,08",$290



Laatste 20 rijen:


,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
1670,Wide World Importers,Regular,Wide World Importers,"High brightness, 1300 ANSI lumen light output,...",626,WWI Projector 480p LCD12 White,19,"$116,75",$229
1671,Wide World Importers,Regular,Wide World Importers,"High brightness, 1300 ANSI lumen light output,...",637,WWI Projector 480p LCD12 Silver,19,"$116,75",$229
1672,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,106,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1673,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,107,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1674,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,108,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1675,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,109,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1676,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,110,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1677,Wide World Importers,Regular,Wide World Importers,"Native 720p resolution, 1700 ANSI lumens, 2500...",614,WWI Projector 720p DLP56 Black,19,"$459,4",$999
1678,Wide World Importers,Regular,Wide World Importers,"Native 720p resolution, 1700 ANSI lumens, 2500...",625,WWI Projector 720p DLP56 White,19,"$459,4",$999
1679,Wide World Importers,Regular,Wide World Importers,"Native 720p resolution, 1700 ANSI lumens, 2500...",636,WWI Projector 720p DLP56 Silver,19,"$459,4",$999


Uit de eerste analyse blijkt dat er 1690 unique keys in de dataset zitten. Dit sluit aan op de eerdere verwachting dat er minimaal 1689 unieke keys aanwezig dienen te zijn vanuit de sales data. Er is ook geen sprake van duplicaten op basis van deze analyse. Uit de weergave blijkt dat producten in verschillende kleuren beschikbaar zijn. Dit kan later interessant zijn voor een analyse op welk kleur product het beste/slechtste verkoopt. 

Gekeken naar de datatypes zou de verwachting zijn dat de UnitCost en Unitprice als floats zouden worden ingeladen. Dit lijkt voorkomen te worden door de dollartekens in de cellen. Dit dit dient opgeschoond te worden. 

In de hele product dataset komen geen lege waarden voor.

De eerste stap is om de kosten en prijs om te zetten naar decimale waarden. Hiermee kan gecontroleerd worden of de kosten en prijs in de product dataset gelijk zijn aan die in de sales dataset.

In [21]:
product_df["UnitCost"] = product_df["UnitCost"].str.removeprefix("$").str.replace(",",".")
product_df["UnitPrice"] = product_df["UnitPrice"].str.removeprefix("$").str.replace(",",".")
#Verwijderen van de $ prefixes uit de kolommen UnitCost en UnitPrice. Daarna de komma vervangen om floats te kunnen maken.

In [22]:
product_df[["UnitCost","UnitPrice"]] = product_df[["UnitCost","UnitPrice"]].apply(pd.to_numeric)
#UnitCost en UnitPrice omzetten in floats.

In [23]:
product_df.info()
#Controleren of waarden zijn omgezet.

<class 'pandas.DataFrame'>
RangeIndex: 1690 entries, 0 to 1689
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   BrandName              1690 non-null   str    
 1   ClassName              1690 non-null   str    
 2   Manufacturer           1690 non-null   str    
 3   ProductDescription     1690 non-null   str    
 4   ProductKey             1690 non-null   int64  
 5   ProductName            1690 non-null   str    
 6   ProductSubcategoryKey  1690 non-null   int64  
 7   UnitCost               1690 non-null   float64
 8   UnitPrice              1690 non-null   float64
dtypes: float64(2), int64(2), str(5)
memory usage: 356.8 KB


In [24]:
product_df.loc[~product_df["ProductKey"].isin(sales_df2["ProductKey"])]
#Zoeken naar producten wel in product dataset maar niet in de salesdata set

,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
35,A. Datum,Economy,A. Datum Corporation,"Digital camera, 3.7 in x 0.9 in x 2",944,A. Datum Compact Digital Camera M200 Black,23,59.32,129.00


De A. Datum Compact Digital Camera M200 Black zit wel in de product dataset maar wordt volgens de sales data niet verkocht in de periode waar deze betrekking op heeft

In [25]:
display(product_df.nunique())
#Aantal unieke eigenschappen onderzoeken.

BrandName                  10
ClassName                   3
Manufacturer               10
ProductDescription        516
ProductKey               1690
ProductName              1690
ProductSubcategoryKey      22
UnitCost                  397
UnitPrice                 350
dtype: int64

In [26]:
#Onderzoeken van strings op verschillende soorten en eventuele dubbel aanwezige gegevens met afwijkende schrijfwijzes.

display(product_df["BrandName"].value_counts())
#Onderzoeken producten van ieder merk
display(product_df["ClassName"].value_counts())
#Onderzoeken segmenten
display(product_df["Manufacturer"].value_counts())
#Onderzoeken Producenten
display(product_df["ProductDescription"].value_counts())
#Onderzoeken productomschrijvingen
display(product_df["ProductName"].value_counts())
#Onderzoeken productnamen

BrandName
Contoso                 560
Proseware               177
Southridge Video        170
Fabrikam                163
The Phone Company       152
Wide World Importers    133
A. Datum                132
Adventure Works         128
Litware                  48
Northwind Traders        27
Name: count, dtype: int64

ClassName
Regular    930
Economy    516
Deluxe     244
Name: count, dtype: int64

Manufacturer
Contoso, Ltd            560
Proseware, Inc.         177
Southridge Video        170
Fabrikam, Inc.          163
The Phone Company       152
Wide World Importers    133
A. Datum Corporation    132
Adventure Works         128
Litware, Inc.            48
Northwind Traders        27
Name: count, dtype: int64

ProductDescription
1080i/p or 720p high definition or standard definition                                             35
6-hour rechargeable battery, 180-degree swivel and flip screen                                     30
Store your CDs or DVDs in these beautiful binders, designed to look like books                     24
Digital camera - SLR with live view mode                                                           21
720p high definition or VGA                                                                        20
                                                                                                   ..
16:9 Aspect Ratio - 113in.Dual wall and ceiling - perfect for home / business.                      1
16:9 Aspect Ratio - 125in. 160 degree wide viewing angle.                                           1
16:9 aspect ratio-113in, dual wall and ceiling - perfect for home / business                        1
2.3GHz dual processor, 4 GB DDR2 SDRAM memories, 4MB L2 Cache  

ProductName
A. Datum Consumer Digital Camera E100 Black             1
A. Datum Consumer Digital Camera E100 Grey              1
A. Datum Consumer Digital Camera E100 Pink              1
A. Datum Consumer Digital Camera E100 Silver            1
A. Datum Consumer Digital Camera E100 Orange            1
                                                       ..
WWI LCD20 M220 White                                    1
WWI Wireless Bluetooth Stereo Headphones M170 Silver    1
WWI Wireless Bluetooth Stereo Headphones M170 Black     1
WWI Wireless Bluetooth Stereo Headphones M170 White     1
WWI Wireless Bluetooth Stereo Headphones M170 Pink      1
Name: count, Length: 1690, dtype: int64

De verschillende merken, segmenten, producenten, productomschrijvingen en productnamen zijn geanalyseerd.
Hier komen geen dubbele waarde in voor veroorzaakt door afwijkende schrijfwijzen.   

Opvallend is het feit dat er **516** unieke omschrijvingen zijn van producten, maar **1690** unieke productbenamingen.
Dit wordt met name veroorzaakt door het feit dat iedere product meerdere kleurvarianten heeft.   

Het lijkt aan te raden om een scheiding te maken tussen productnamen en kleuren. 
Zo kan er zuiver gekeken worden naar productverkopen en later naar de voorkeur van de kleur. 
De verwachting is dat na opschonen het aantal unieke productnamen zonder kleur gelijk is aan de unieke productomschrijvingen.

De volgende stap is om te kijken of product kosten en prijzen overeenkomen met de gegevens uit de sales set.

In [27]:
sales_df2["ProductUnitCost"] = sales_df2["ProductKey"].map(
    product_df.set_index("ProductKey")["UnitCost"]
)
#De Unitcost van de productdataset koppelen aan de sales_df2 dataset.

sales_df["CostControl"] = (
    sales_df2["ProductUnitCost"] == sales_df2["UnitCost"]
)
#Controleren of ProductUnitCost in product gelijk is aan Unitcost in de sales dataset.

In [28]:
sales_df[~sales_df["CostControl"]]
#Controleren waar UnitCost niet overeenkomt.

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,CostControl


In [29]:
sales_df["ProductUnitPrice"] = sales_df["ProductKey"].map(
    product_df.set_index("ProductKey")["UnitPrice"]
)
#De UnitPrice van de productdataset koppelen aan de sales_df2 dataset.
sales_df["PriceControl"] = (
    sales_df["ProductUnitPrice"] == sales_df["UnitPrice"]
)
#Controleren of 

In [30]:
sales_df[~sales_df["PriceControl"]]
#Onderzoeken waar prijs niet overeenkomt.

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,CostControl,ProductUnitPrice,PriceControl


Uit vergelijkingen met de sales dataset blijkt dat kosten en prijzen voor alle sales overeenkomen met die uit de productdata.

### **Tussentijdse bevindingen**

Conform eerdere verwachtingen bevat de product dataset minimaal 1689 productkeys, het totaal is 1690. 
Het enige product dat niet voorkomt in de sales dataset is de *A. Datum Comapct Digital Camera M200 black*.  
Verder is er gekeken naar volgende eigenschappen:
- Merknamen: 10 
- Segmenten: 3 
- Producten: 10 
- Unieke productomschrijvingen: 516 
- Unieke productnamen: 1690 

Er is in deze benamingen geen besprake van duplicaten ontstaan door bijvoorbeeld verschillende schrijfwijzes.  
Het is wel opvallend dat er 516 productomschrijvingen zijn, tegenover 1690 productnamen. Nadere analyse toont aan dat dit wordt veroorzaakt door het feit dat producten verkrijgbaar zijn in meerdere kleuren. 

*To-do*:
- Aparte kolommen toevoegen voor productnaam en kleur.

## 2.3 Splitsen kolommen

In [31]:
product_df["ProductColor"] = product_df["ProductName"].str.split().str[-1]
#Product kleuren opslaan naar ProductKolom.
product_df["ProductColor"].value_counts()
#Controleren op waarden en aantallen.

ProductColor
Black     485
White     356
Silver    261
Grey      195
Blue       70
Brown      64
Pink       56
Gold       47
Orange     42
Green      40
Red        39
Yellow     15
Azure      14
Purple      3
blue        2
E120        1
Name: count, dtype: int64

Eerst converise laat zien dat dat er 2 verschillende wijze zijn voor blue. Eén met en één zonder hoofdletter. Daarnaast is er één waarde waarvoor de regel dat het laatste de woord de kleur is, niet op gaat. 

In [32]:
product_df["ProductColor"]=product_df["ProductColor"].str.title()
#Schrijfwijze veranderen naar title, eerst letter is een hoofdletter, de rest kleine letters.
product_df["ProductColor"].value_counts()
#Opnieuw tellen. Enkel het probleem met de E120 blijft.
product_df.loc[product_df["ProductColor"]=="E120"]
#Er staat geen kleur  bij deze waarde, dus de standaard splits wijze gaat niet op.

,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice,ProductColor
421,Contoso,Economy,"Contoso, Ltd",Expect better quality than any other screen pr...,2494,Reusable Phone Screen Protector E120,33,1.50,2.94,E120


In [33]:
product_df.loc[(product_df["ProductColor"]=="E120"),"ProductColor"]="No Color"
#Waarde zoeken in color kolom en vervangen door "No Color", zodat later bekend is dat deze geen kleur heeft.
display(product_df["ProductColor"].value_counts())
display(product_df["ProductColor"].value_counts().sum())

ProductColor
Black       485
White       356
Silver      261
Grey        195
Blue         72
Brown        64
Pink         56
Gold         47
Orange       42
Green        40
Red          39
Yellow       15
Azure        14
Purple        3
No Color      1
Name: count, dtype: int64

np.int64(1690)

Alle productregels hebben nu een eigen kleur kolom. 1 product heeft geen kleur.
Nu dienen de producten nog gesplitst te worden.

In [34]:
product_df["ProductNameNoColor"] = product_df["ProductName"].str.split().str[:-1].str.join(" ")
#Splitsen productnamen van kleuren, zodat er één productnaam overblijft, op basis van de aanname dat het laatste woord de kleur betreft.
#split maakt een lijst van woorden, het laatste woord wordt verwijderd, de string wordt weer samengevoegd en zou de productnaam zonder kleur moeten weergeven.
product_df[product_df["ProductNameNoColor"].str.split().str[-1].isin(product_df["ProductColor"])]
#Controleren of laatste waarde van de productnaam zonder kleur in de kleur kolom zit.


,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice,ProductColor,ProductNameNoColor
28,A. Datum,Economy,A. Datum Corporation,"Digital camera - SLR with live view mode, 5 in...",1058,A. Datum SLR Camera M138 Silver Grey,24,143.48,312.00,Grey,A. Datum SLR Camera M138 Silver
44,A. Datum,Regular,A. Datum Corporation,"5 in x 2.4 in x 3.9 in, 15.9 oz, digital camer...",1059,"A. Datum SLR Camera 35"" M358 Silver Grey",24,155.43,338.00,Grey,"A. Datum SLR Camera 35"" M358 Silver"
56,A. Datum,Regular,A. Datum Corporation,"Digital camera – compact, 3.8 in x 1 in x 2.2 ...",1042,A. Datum Point Shoot Digital Camera M500 Silve...,23,91.05,198.00,Grey,A. Datum Point Shoot Digital Camera M500 Silver
111,A. Datum,Regular,A. Datum Corporation,Dynamic integrated dust reduction system,1056,A. Datum SLR Camera X138 Silver Grey,24,207.74,627.00,Grey,A. Datum SLR Camera X138 Silver
113,A. Datum,Regular,A. Datum Corporation,Fast startup and split-second shutter response...,1057,"A. Datum SLR Camera 35"" X358 Silver Grey",24,194.82,588.00,Grey,"A. Datum SLR Camera 35"" X358 Silver"
122,A. Datum,Regular,A. Datum Corporation,"Optical zoom 2.7in LCD digital camera, 100kgf",1043,A. Datum Full Frame Digital Camera X300 Silver...,23,76.53,231.00,Grey,A. Datum Full Frame Digital Camera X300 Silver
297,Contoso,Deluxe,"Contoso, Ltd",Dynamic integrated dust reduction system,1088,Contoso SLR Camera X144 Silver Grey,24,222.98,673.00,Grey,Contoso SLR Camera X144 Silver
638,Contoso,Regular,"Contoso, Ltd",5 in x 2.4 in x 3.9 in. 15.9 oz. Digital camer...,1091,"Contoso SLR Camera 35"" M358 Silver Grey",24,164.63,358.00,Grey,"Contoso SLR Camera 35"" M358 Silver"
708,Contoso,Regular,"Contoso, Ltd","Digital camera - SLR with live view mode, 5 in...",1090,Contoso SLR Camera M144 Silver Grey,24,152.68,332.00,Grey,Contoso SLR Camera M144 Silver
735,Contoso,Regular,"Contoso, Ltd",Fast startup and split-second shutter response...,1089,"Contoso SLR Camera 35"" X358 Silver Grey",24,188.19,568.00,Grey,"Contoso SLR Camera 35"" X358 Silver"


Door de foutieve aanname date alle kleuren 1 woord zijn is silver Grey nu niet goed in de kleur kolom gezet. Dit dient gecorrigeerd te worden alvorens verder te gaan. 

In [35]:
product_df.loc[product_df["ProductName"].str.contains("Silver Grey"),"ProductColor"] = "Silver Grey"
#Zoek de productnamen met "Silver Grey" en zet de waarde voor ProductColor naar Silver Grey.

In [36]:
kleuren = list(product_df["ProductColor"].unique())
#maak een list van kleuren.
masker = product_df["ProductName"].apply(lambda naam: any(kleur in naam for kleur in kleuren))
#Masker dat controleert of kleur in de productnaam voor komt.
display(product_df[~masker])
#Laat de producten zien waarvoor dit niet van toepassing is. Dit zijn de producten die we eerder gecorrigeerd hebben.


,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice,ProductColor,ProductNameNoColor
421,Contoso,Economy,"Contoso, Ltd",Expect better quality than any other screen pr...,2494,Reusable Phone Screen Protector E120,33,1.50,2.94,No Color,Reusable Phone Screen Protector
444,Contoso,Economy,"Contoso, Ltd","Illuminate wireless reading device, two super ...",843,Contoso Bright Light battery E20 blue,22,13.71,26.90,Blue,Contoso Bright Light battery E20
599,Contoso,Regular,"Contoso, Ltd","2.4'' LCD Touch screen, 8GB flash memory, play...",44,Contoso 16GB New Generation MP5 Player M1650 blue,1,106.69,232.00,Blue,Contoso 16GB New Generation MP5 Player M1650


Kleur lijkt nu opgelost te zijn. Nu een zuivere productnaam creeëren.

In [37]:
product_df["ProductNameNoColor"] = product_df.apply(
    lambda row: row["ProductName"].removesuffix(row["ProductColor"]).strip(),
    axis=1
)
#Nieuwe functie die met behulp van apply voor iedere rij een nieuwe waarde berekent die de productcolor verwijdert van de originele productname.


In [38]:
product_df["ProductNameNoColor"] = product_df.apply(
    # Voert de onderstaande lambda-functie uit op iedere rij van product_df.
    lambda row: re.sub(
        # Vervangt tekst op basis van een regex-patroon.
        rf"{re.escape(row['ProductColor'])}$",
        # Bouwt het zoekpatroon op uit de kleur van de huidige rij.
        # re.escape() zorgt dat eventuele speciale regextekens in de kleur veilig worden behandeld.
        # $ betekent: de kleur moet aan het einde van ProductName staan.
        "",
        # Vervangt de gevonden kleur door een lege string.
        row["ProductName"],
        # Dit is de tekst waarin gezocht en vervangen wordt.
        flags=re.IGNORECASE
        # Negeert hoofdletters en kleine letters.
        # Dus "Blue", "blue" en "BLUE" worden allemaal gevonden.
    ).strip(),
    # Verwijdert eventuele overgebleven spaties aan het begin of einde.
    axis=1
    # Geeft aan dat apply rij voor rij moet werken in plaats van kolom voor kolom.
)

In [39]:
Duplicatecontrole = product_df[["ProductNameNoColor","ProductDescription"]].drop_duplicates()
#Dataframe makenn met alleen productnamen en descriptions.
Duplicatecontrole["ProductDescription"].value_counts()
#Aantal keren tellen dat waarden voorkomen. 


ProductDescription
1080i/p or 720p high definition or standard definition                                                        20
Digital camera - SLR with live view mode                                                                      12
6-hour rechargeable battery, 180-degree swivel and flip screen                                                10
DVD and VHS recorder with two way dubbing, records -R/-RW, +R/+RW formats                                     10
720p high definition or VGA                                                                                    9
                                                                                                              ..
2.3GHz dual processor, 4 GB DDR2 SDRAM memories, 4MB L2 Cache                                                  1
2GB flash memory, USB driver, continuous recording until memory full, as spy cameras, hidden cameras           1
3.0GHz processor, 4 GB DDR2 SDRAM memories, 500GB 7200RPM hard drive, 19 inch

De dataframe met alleen productnames en descriptions geeft 516 unieke descriptions met maximaal 20 keer hergebruikt en minimaal 1 keer.
Dit bevestigt dus dat er voor meerdere producten dezelfde description wordt gebruikt.  

In [40]:
product_df.to_parquet(r"PQ/Contoso_product.parquet")

# 3 ***Analyse calendar data***
In dit onderdeel kijken we naar de calendar dataset.
Naar verwachting zal deze matchen op de datekey en minimaal datums van 1 januari 2011 tot en met 31 december 2013 bevatten.

### 3.1 Importeren dataset
Met behulp van path en pandas importeren wordt de dataframe geïmporteerd voor analyse.

In [41]:
calendar_path= Path(r"Data\Contoso_calendar.csv")
#Inlezen calendar.csv als raw string om speciale tekens te escapen.
calendar_df = pd.read_csv(calendar_path,
                       decimal=",")

### 3.2 Analyse dataset
Hier voeren we de eerste analyse uit op de calendar dataset.

In [42]:
init_analyse(calendar_df)
#Met behulp van analyse functie eerste algemene gegeens opvragen.

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 5113 entries, 0 to 5112
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   DateInt        5113 non-null   int64
 1   Year           5113 non-null   int64
 2   Quarter        5113 non-null   str  
 3   Month          5113 non-null   str  
 4   Day            5113 non-null   int64
 5   DayOfMonth     5113 non-null   int64
 6   DayOfWeekName  5113 non-null   str  
 7   MonthName      5113 non-null   str  
 8   MonthOfYear    5113 non-null   int64
 9   QuarterOfYear  5113 non-null   int64
 10  Year.1         5113 non-null   int64
dtypes: int64(7), str(4)
memory usage: 561.6 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


DateInt          0
Year             0
Quarter          0
Month            0
Day              0
DayOfMonth       0
DayOfWeekName    0
MonthName        0
MonthOfYear      0
QuarterOfYear    0
Year.1           0
dtype: int64


Aantal unieke waarden:


DateInt          5113
Year               14
Quarter             4
Month              12
Day                31
DayOfMonth         31
DayOfWeekName       7
MonthName          12
MonthOfYear        12
QuarterOfYear       4
Year.1             14
dtype: int64


Eerste 20 rijen:


,DateInt,Year,Quarter,Month,Day,DayOfMonth,DayOfWeekName,MonthName,MonthOfYear,QuarterOfYear,Year.1
0,20050101,2005,Qtr 1,January,1,1,Saturday,January,1,1,2005
1,20050102,2005,Qtr 1,January,2,2,Sunday,January,1,1,2005
2,20050103,2005,Qtr 1,January,3,3,Monday,January,1,1,2005
3,20050104,2005,Qtr 1,January,4,4,Tuesday,January,1,1,2005
4,20050105,2005,Qtr 1,January,5,5,Wednesday,January,1,1,2005
5,20050106,2005,Qtr 1,January,6,6,Thursday,January,1,1,2005
6,20050107,2005,Qtr 1,January,7,7,Friday,January,1,1,2005
7,20050108,2005,Qtr 1,January,8,8,Saturday,January,1,1,2005
8,20050109,2005,Qtr 1,January,9,9,Sunday,January,1,1,2005
9,20050110,2005,Qtr 1,January,10,10,Monday,January,1,1,2005



Laatste 20 rijen:


,DateInt,Year,Quarter,Month,Day,DayOfMonth,DayOfWeekName,MonthName,MonthOfYear,QuarterOfYear,Year.1
5093,20181212,2018,Qtr 4,December,12,12,Wednesday,December,12,4,2018
5094,20181213,2018,Qtr 4,December,13,13,Thursday,December,12,4,2018
5095,20181214,2018,Qtr 4,December,14,14,Friday,December,12,4,2018
5096,20181215,2018,Qtr 4,December,15,15,Saturday,December,12,4,2018
5097,20181216,2018,Qtr 4,December,16,16,Sunday,December,12,4,2018
5098,20181217,2018,Qtr 4,December,17,17,Monday,December,12,4,2018
5099,20181218,2018,Qtr 4,December,18,18,Tuesday,December,12,4,2018
5100,20181219,2018,Qtr 4,December,19,19,Wednesday,December,12,4,2018
5101,20181220,2018,Qtr 4,December,20,20,Thursday,December,12,4,2018
5102,20181221,2018,Qtr 4,December,21,21,Friday,December,12,4,2018


De dataset is groter dan de sales dataset qua datumbereik. Dit is opzich geen probleem, echter kan dit wel voor extra load zorgen gezien gegevens die niet worden gebruikt worden ingeladen.
Een actiepunt is om eventueel later te kijken naar het schrappen van datums die niet relevant zijn om performance te verbeteren.
Alle waarden zijn gevuld en volgen een standaard structuur. 
De Year kolom lijkt 2 keer aanwezig, eventueel kan er één gedropt worden uit de set. Daarnaast lijken Day en DayOfMonth dezelfde gegevens te weergeven. 

Op het moment is het niet zeker of deze tabel in PowerBI direct werkt, voor pandas gebruik hebben we een kolom nodig die als datetime functioneert. 
De volgende stap is om op voorhand een kolom te maken die als datetime gebruikt kan worden. 

In [43]:
calendar_df["DateInt"].isin(sales_df["DateKey"])
#Controleren matches in sales dataset.

0       False
1       False
2       False
3       False
4       False
        ...  
5108    False
5109    False
5110    False
5111    False
5112    False
Name: DateInt, Length: 5113, dtype: bool

In de sales data wordt gebruik gemaakt van een DateKey kolom. Deze waarde komt niet terug in de calendar dataset en matcht ook niet op DateInt.
Volgende stap is  het aanmaken van een key die matcht op de key in de sales set om datums te verbinden.

### 3.3 Datetime value maken
In dit onderdeel maken we met de aanwezige gegevens een DateKey om te matchen op de waarde in de sales set.

In [44]:
calendar_df.head(20)

,DateInt,Year,Quarter,Month,Day,DayOfMonth,DayOfWeekName,MonthName,MonthOfYear,QuarterOfYear,Year.1
0,20050101,2005,Qtr 1,January,1,1,Saturday,January,1,1,2005
1,20050102,2005,Qtr 1,January,2,2,Sunday,January,1,1,2005
2,20050103,2005,Qtr 1,January,3,3,Monday,January,1,1,2005
3,20050104,2005,Qtr 1,January,4,4,Tuesday,January,1,1,2005
4,20050105,2005,Qtr 1,January,5,5,Wednesday,January,1,1,2005
5,20050106,2005,Qtr 1,January,6,6,Thursday,January,1,1,2005
6,20050107,2005,Qtr 1,January,7,7,Friday,January,1,1,2005
7,20050108,2005,Qtr 1,January,8,8,Saturday,January,1,1,2005
8,20050109,2005,Qtr 1,January,9,9,Sunday,January,1,1,2005
9,20050110,2005,Qtr 1,January,10,10,Monday,January,1,1,2005


In [45]:
calendar_df["DateKey"] = pd.to_datetime(calendar_df["DateInt"], format="%Y%m%d")
#Datekey aanmaken in calendar set door de DateInt te converteren naar een datetime op basis van het format.


In [46]:
calendar_df[calendar_df["DateKey"].isin(sales_df2["DateKey"])]
#Controleren of de DateKey matcht op waarden in de sales data. 

,DateInt,Year,Quarter,Month,Day,DayOfMonth,DayOfWeekName,MonthName,MonthOfYear,QuarterOfYear,Year.1,DateKey
2191,20110101,2011,Qtr 1,January,1,1,Saturday,January,1,1,2011,2011-01-01
2192,20110102,2011,Qtr 1,January,2,2,Sunday,January,1,1,2011,2011-01-02
2193,20110103,2011,Qtr 1,January,3,3,Monday,January,1,1,2011,2011-01-03
2194,20110104,2011,Qtr 1,January,4,4,Tuesday,January,1,1,2011,2011-01-04
2195,20110105,2011,Qtr 1,January,5,5,Wednesday,January,1,1,2011,2011-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...
3282,20131227,2013,Qtr 4,December,27,27,Friday,December,12,4,2013,2013-12-27
3283,20131228,2013,Qtr 4,December,28,28,Saturday,December,12,4,2013,2013-12-28
3284,20131229,2013,Qtr 4,December,29,29,Sunday,December,12,4,2013,2013-12-29
3285,20131230,2013,Qtr 4,December,30,30,Monday,December,12,4,2013,2013-12-30


Met behulp van de DateIn waarde is een DateKey aangemaakt die matcht op gegevens in de sales dataset. 
Bij Controle van de dataset start deze op 1 januari 2011 en eindigt deze op 31 december 2013. Dit komt overeen met de sales dataset.
Dit lijkt voornamelijk relevant voor het PowerBI gedeelte.

## **Voorlopige conclusie**

De calendar data lijkt met name voor de tijdfunctie binnen PowerBI.
Voor de zekerheid is een key aangemaakt om op te matchen met de sales data.  
Day en DayOfMonth lijken dezelfde waarde te bevatten, idem voor Year en Year.1

*actiepunten*:
- Eventueel dubbele waarden voor day en year laten vallen.
- Tabel verkleinen tot bereik van de dataset.

## Opschonen dataset

In [47]:
calendar_df = calendar_df.drop(["Day","Year.1"],axis=1)

In [48]:
calendar_df

,DateInt,Year,Quarter,Month,DayOfMonth,DayOfWeekName,MonthName,MonthOfYear,QuarterOfYear,DateKey
0,20050101,2005,Qtr 1,January,1,Saturday,January,1,1,2005-01-01
1,20050102,2005,Qtr 1,January,2,Sunday,January,1,1,2005-01-02
2,20050103,2005,Qtr 1,January,3,Monday,January,1,1,2005-01-03
3,20050104,2005,Qtr 1,January,4,Tuesday,January,1,1,2005-01-04
4,20050105,2005,Qtr 1,January,5,Wednesday,January,1,1,2005-01-05
...,...,...,...,...,...,...,...,...,...,...
5108,20181227,2018,Qtr 4,December,27,Thursday,December,12,4,2018-12-27
5109,20181228,2018,Qtr 4,December,28,Friday,December,12,4,2018-12-28
5110,20181229,2018,Qtr 4,December,29,Saturday,December,12,4,2018-12-29
5111,20181230,2018,Qtr 4,December,30,Sunday,December,12,4,2018-12-30


Dubbele waarden zijn verwijderd en er is een werkende datekey gemaakt.

In [49]:
calendar_df2 = calendar_df[calendar_df["DateKey"].between("01-01-2011","31-12-2014")].reset_index()
#Een nieuwe calendar dataset met alleen datums relevant tot de dataset om performance optimaal te houden.

In [50]:
calendar_df2.to_parquet(r"PQ/Contoso_calendar.parquet")

# 4 ***Analyse geography data***
In dit onderdeel wordt de product dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data set. 

### 4.1 Importeren dataset


In [51]:
geography_path= Path(r"Data\Contoso_Geography.csv")
#Inlezen geography.csv
geography_df = pd.read_csv(geography_path,
                       decimal=",")

### 4.2 Analyse dataset

In [52]:
init_analyse(geography_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 674 entries, 0 to 673
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   ContinentName      674 non-null    str  
 1   GeographyKey       674 non-null    int64
 2   GeographyType      674 non-null    str  
 3   RegionCountryName  671 non-null    str  
dtypes: int64(1), str(3)
memory usage: 38.2 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


ContinentName        0
GeographyKey         0
GeographyType        0
RegionCountryName    3
dtype: int64


Aantal unieke waarden:


ContinentName          3
GeographyKey         674
GeographyType          4
RegionCountryName     34
dtype: int64


Eerste 20 rijen:


,ContinentName,GeographyKey,GeographyType,RegionCountryName
0,Asia,1,Continent,NaN
1,Asia,269,Country/Region,Armenia
2,Asia,270,Country/Region,Australia
3,Asia,271,Country/Region,Bhutan
4,Asia,272,Country/Region,China
5,Asia,273,Country/Region,India
6,Asia,274,Country/Region,Iran
7,Asia,275,Country/Region,Japan
8,Asia,276,Country/Region,Kyrgyzstan
9,Asia,277,Country/Region,Pakistan



Laatste 20 rijen:


,ContinentName,GeographyKey,GeographyType,RegionCountryName
654,North America,933,City,United States
655,North America,934,City,United States
656,North America,935,City,United States
657,North America,936,City,United States
658,North America,937,City,United States
659,North America,938,City,United States
660,North America,939,City,United States
661,North America,940,City,United States
662,North America,941,City,United States
663,North America,942,City,Canada


De dataset lijkt opgebouwd uit contintenten, landen, provincies/regios en steden.  
Bij regioncountryname zijn er 3 lege waarden gevonden in de initiële analyse. Hier dient op ingezoomd te worden. Er is geen sprake van duplicated.

In [53]:
geography_df[geography_df["RegionCountryName"].isna()]
#Waarden met ontbrekende regioncountryname bekijken.

,ContinentName,GeographyKey,GeographyType,RegionCountryName
0,Asia,1,Continent,NaN
114,Europe,2,Continent,NaN
335,North America,3,Continent,NaN


De lege waarden lijken betrekking hebben op het hoogste niveau, contintent. Dit lijkt logisch, eventueel kan hier later een mock waarde ingegeven worden. 

In [54]:
geographystr = ["ContinentName","GeographyType","RegionCountryName"]

for g in geographystr:
    display(geography_df[g].value_counts())
#Onderzoeken dataset op afwijkende schrijfwijzes, daarnaast de samenstelling van de geography set.

ContinentName
North America    339
Europe           221
Asia             114
Name: count, dtype: int64

GeographyType
City              517
State/Province    120
Country/Region     34
Continent           3
Name: count, dtype: int64

RegionCountryName
United States      312
France              65
Germany             54
Australia           47
United Kingdom      47
Canada              26
Japan               13
China               11
Italy               11
India                7
Russia               7
South Korea          5
Armenia              3
Bhutan               3
Iran                 3
Kyrgyzstan           3
Pakistan             3
Singapore            3
Syria                3
Taiwan               3
Thailand             3
Turkmenistan         3
Denmark              3
Greece               3
the Netherlands      3
Ireland              3
Malta                3
Poland               3
Portugal             3
Romania              3
Slovenia             3
Spain                3
Sweden               3
Switzerland          3
Name: count, dtype: int64

In [55]:
geography_df.to_parquet(r"PQ/Contoso_geography.parquet")

## Voorlopige conclusie
De geography dataset bevate gegevens omtrent locaties. Deze kunnen niet direct gelinkt worden aan de sales dataset.  
De voorlopige aanname is dat deze samenhangt met store data.
De gegevens hebben met name betrekking op de Verenigde Staten, daarop volgt Europa en uiteindelijk Europa.

Het zou interessant kunnen zijn om ontwikkelingen van de verkopen op deze verschillende contintenten te zien en vergelijken. 
Eventueel product context.

# 5 ***Analyse store data***
In dit onderdeel wordt de product dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales dataset en de geography dataset

### 5.1 Data importeren

In [56]:
stores_path= Path(r"Data\Contoso_Stores.csv")
#Inlezen stores.csv
stores_df = pd.read_csv(stores_path,
                       decimal=",")

### 5.2 Data analyseren

In [57]:
init_analyse(stores_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CloseReason      12 non-null     str    
 1   EmployeeCount    305 non-null    float64
 2   GeographyKey     306 non-null    int64  
 3   SellingAreaSize  306 non-null    int64  
 4   Status           306 non-null    str    
 5   StoreKey         306 non-null    int64  
 6   StoreName        306 non-null    str    
 7   StoreType        306 non-null    str    
dtypes: float64(1), int64(3), str(4)
memory usage: 28.5 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


CloseReason        294
EmployeeCount        1
GeographyKey         0
SellingAreaSize      0
Status               0
StoreKey             0
StoreName            0
StoreType            0
dtype: int64


Aantal unieke waarden:


CloseReason          2
EmployeeCount       46
GeographyKey       263
SellingAreaSize     15
Status               2
StoreKey           306
StoreName          306
StoreType            4
dtype: int64


Eerste 20 rijen:


,CloseReason,EmployeeCount,GeographyKey,SellingAreaSize,Status,StoreKey,StoreName,StoreType
0,NaN,19.00,424,455,On,4,Contoso Bellevue Store,Store
1,NaN,26.00,449,680,On,156,Contoso Cambridge Store,Store
2,NaN,19.00,453,455,On,88,Contoso Cedar Park Store,Store
3,NaN,25.00,501,680,On,214,Contoso Leeds Store,Store
4,NaN,17.00,506,462,On,201,Contoso London Store,Store
5,NaN,12.00,529,680,On,215,Contoso Paris Store,Store
6,NaN,12.00,529,551,On,309,Contoso Europe Reseller,Reseller
7,NaN,25.00,551,700,On,17,Contoso Spokane Store,Store
8,NaN,20.00,558,480,On,253,Contoso Sydney No.1 Store,Store
9,NaN,95.00,558,480,On,300,Contoso Sydney No.2 Store,Store



Laatste 20 rijen:


,CloseReason,EmployeeCount,GeographyKey,SellingAreaSize,Status,StoreKey,StoreName,StoreType
286,NaN,19.00,946,455,On,181,Contoso Virginia Beach Store,Store
287,NaN,25.00,947,700,On,10,Contoso Wapato Store,Store
288,NaN,33.00,948,560,On,174,Contoso Waterbury Store,Store
289,NaN,19.00,949,455,On,50,Contoso Waukesha No.1 Store,Store
290,NaN,17.00,949,462,On,54,Contoso Waukesha No.2 Store,Store
291,NaN,47.00,950,1125,On,21,Contoso Wheat Ridge Store,Store
292,NaN,33.00,951,560,On,182,Contoso Winchester Store,Store
293,NaN,25.00,952,700,On,147,Contoso Worcester No.1 Store,Store
294,Relocation,22.00,783,500,Off,184,Contoso Alexandria Store,Store
295,Relocation,22.00,851,500,Off,84,Contoso Humble Store,Store


Volgens de dataset zijn er 306 unieke storekeys dit komt overeen met eerdere analyse uit de sales data.
Met name in de CloseReason kolom zijn veel lege waarden. De aanname is dat deze pas wordt ingevuld bij een sluiting van een locatie.
Er is geen data aanwezig over wanneer de winkel is gesloten, het lijkt dus niet mogelijk te oordelen of er foutieve toewijzigingen aan een winkel zijn in de salessset.
Storetypes lijkt samen te kunnen hangen met de verkoopkanalen die voort komen uit de sales dataset.

Daarnaast ontbreekt er bij 1 store een employeecount, hier dient naar gekeken te worden. Tevens worden employeecounts op het moment als floats opgeslagen, ervanuit gaande dat dit om personen gaat en niet FTE.

Zoals eerder verwacht is de geographykey verwerkt in deze dataset.

In [58]:
Control = ["CloseReason","Status","StoreName","StoreType"]
display(stores_df.nunique())

for c in Control:
    display(stores_df[c].value_counts())
#Loop om values te weergeven bij unieke waardes.



CloseReason          2
EmployeeCount       46
GeographyKey       263
SellingAreaSize     15
Status               2
StoreKey           306
StoreName          306
StoreType            4
dtype: int64

CloseReason
Relocation    10
Store          2
Name: count, dtype: int64

Status
On     294
Off     12
Name: count, dtype: int64

StoreName
Contoso Bellevue Store          1
Contoso Cambridge Store         1
Contoso Cedar Park Store        1
Contoso Leeds Store             1
Contoso London Store            1
                               ..
Contoso Stoughton Store         1
Contoso Trenton No.2 Store      1
Contoso Worcester No.2 Store    1
Contoso Buffalo Store           1
Contoso Cheney Store            1
Name: count, Length: 306, dtype: int64

StoreType
Store       299
Reseller      3
Online        3
Catalog       1
Name: count, dtype: int64

Volgende feiten vallen op gekeken naar de verschillende omschrijvingen en namen:
- 2 sluitingsreden
- 263 geographykeys t.o.v. 674 in de geography set
- 15 selling area sizes, de vraag is of dit daadwerkelijke oppervlakte is of een soort bucket?
- 4 storetypes, lijkt dus gelijk te zijn aan de verkoopkanalen.
 

In [59]:
stores_df["StoreKey"].isin(sales_df2["StoreKey"])
sales_df2["StoreKey"].isin(stores_df["StoreKey"])
#Checken of alle waarden aanwezig zijn in salesset

0          True
1          True
2          True
3          True
4          True
           ... 
2282477    True
2282478    True
2282479    True
2282480    True
2282481    True
Name: StoreKey, Length: 2282482, dtype: bool

In [60]:
display(geography_df[~geography_df["GeographyKey"].isin(stores_df["GeographyKey"])])
#Waarden niet in store set.

display(geography_df[geography_df["GeographyKey"].isin(stores_df["GeographyKey"])])
#Waarden in de store set.

,ContinentName,GeographyKey,GeographyType,RegionCountryName
0,Asia,1,Continent,NaN
1,Asia,269,Country/Region,Armenia
2,Asia,270,Country/Region,Australia
3,Asia,271,Country/Region,Bhutan
4,Asia,272,Country/Region,China
...,...,...,...,...
498,North America,694,City,United States
499,North America,695,City,Canada
500,North America,701,City,Canada
501,North America,703,City,United States


,ContinentName,GeographyKey,GeographyType,RegionCountryName
65,Asia,558,City,Australia
86,Asia,708,City,Turkmenistan
87,Asia,709,City,Thailand
88,Asia,710,City,China
89,Asia,711,City,Kyrgyzstan
...,...,...,...,...
669,North America,948,City,United States
670,North America,949,City,United States
671,North America,950,City,United States
672,North America,951,City,United States


## Voorlopige Conclusie
Storekeys in de salesset komen overeen met de sales set. 
Een groot aantal geography locaties komen niet voor in de storeset.
Dit betekent niet dat ze niet relevant zijn, hier kunnen verkopen zijn via andere kanalen. 
Dit analyseren aan de hand van de channel dataset.

Wellicht interessant om in kaart te brengen of er locaties/regio's zijn waar veel verkopen zijn via andere kanalen waar een store van toegevoegde waarde zou zijn.

Er zijn geen historische gegevens beschikbaar tot en met de relocates. Daarnaast is de CloseReason "Store" niet geheel duidelijk.

In [61]:
stores_df.to_parquet(r"PQ/Contoso_stores.parquet")

# 6 ***Analyse Promotions data***
In dit onderdeel wordt de promotions dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data.

### 6.1 Data importeren

In [62]:
promotions_path= Path(r"Data\Contoso_Promotion.csv")
#Inlezen promotions.csv
promotions_df = pd.read_csv(promotions_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.


### 6.2 Data analyse

In [63]:
init_analyse(promotions_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   DiscountPercent  28 non-null     float64
 1   Year             28 non-null     int64  
 2   Quarter          28 non-null     str    
 3   Month            28 non-null     str    
 4   Day              28 non-null     int64  
 5   PromotionKey     28 non-null     int64  
 6   PromotionLabel   28 non-null     int64  
 7   PromotionName    28 non-null     str    
 8   Year.1           28 non-null     int64  
 9   Quarter.1        28 non-null     str    
 10  Month.1          28 non-null     str    
 11  Day.1            28 non-null     int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 4.1 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


DiscountPercent    0
Year               0
Quarter            0
Month              0
Day                0
PromotionKey       0
PromotionLabel     0
PromotionName      0
Year.1             0
Quarter.1          0
Month.1            0
Day.1              0
dtype: int64


Aantal unieke waarden:


DiscountPercent     6
Year                4
Quarter             4
Month               6
Day                 2
PromotionKey       28
PromotionLabel     28
PromotionName      10
Year.1              4
Quarter.1           4
Month.1             7
Day.1               1
dtype: int64


Eerste 20 rijen:


,DiscountPercent,Year,Quarter,Month,Day,PromotionKey,PromotionLabel,PromotionName,Year.1,Quarter.1,Month.1,Day.1
0,0.05,2011,Qtr 1,March,31,2,2,North America Spring Promotion,2011,Qtr 1,January,1
1,0.20,2011,Qtr 2,April,30,6,6,Asian Spring Promotion,2011,Qtr 1,February,1
2,0.07,2011,Qtr 2,April,30,8,8,European Spring Promotion,2011,Qtr 1,February,1
3,0.10,2011,Qtr 2,June,30,7,7,Asian Summer Promotion,2011,Qtr 2,May,1
4,0.10,2011,Qtr 3,September,30,3,3,North America Back-to-School Promotion,2011,Qtr 3,July,1
5,0.10,2011,Qtr 3,September,30,9,9,European Back-to-Scholl Promotion,2011,Qtr 3,August,1
6,0.20,2011,Qtr 4,December,31,4,4,North America Holiday Promotion,2011,Qtr 4,November,1
7,0.15,2012,Qtr 1,January,31,5,5,Asian Holiday Promotion,2011,Qtr 4,November,1
8,0.20,2012,Qtr 1,January,31,10,10,European Holiday Promotion,2011,Qtr 4,October,1
9,0.05,2012,Qtr 1,March,31,11,11,North America Spring Promotion,2012,Qtr 1,January,1



Laatste 20 rijen:


,DiscountPercent,Year,Quarter,Month,Day,PromotionKey,PromotionLabel,PromotionName,Year.1,Quarter.1,Month.1,Day.1
8,0.20,2012,Qtr 1,January,31,10,10,European Holiday Promotion,2011,Qtr 4,October,1
9,0.05,2012,Qtr 1,March,31,11,11,North America Spring Promotion,2012,Qtr 1,January,1
10,0.20,2012,Qtr 2,April,30,15,15,Asian Spring Promotion,2012,Qtr 1,February,1
11,0.07,2012,Qtr 2,April,30,17,17,European Spring Promotion,2012,Qtr 1,February,1
12,0.10,2012,Qtr 2,June,30,16,16,Asian Summer Promotion,2012,Qtr 2,May,1
13,0.10,2012,Qtr 3,September,30,12,12,North America Back-to-School Promotion,2012,Qtr 3,July,1
14,0.10,2012,Qtr 3,September,30,18,18,European Back-to-Scholl Promotion,2012,Qtr 3,August,1
15,0.20,2012,Qtr 4,December,31,13,13,North America Holiday Promotion,2012,Qtr 4,November,1
16,0.15,2013,Qtr 1,January,31,14,14,Asian Holiday Promotion,2012,Qtr 4,November,1
17,0.20,2013,Qtr 1,January,31,19,19,European Holiday Promotion,2012,Qtr 4,October,1


DiscountPercent is in tegenstelling tot voorgaande datasets met een . voor decimalen aangegeven. 
Er zijn 2 datum waarden aanwezig in de set. Een eerste analyse lijkt er op te wijzen dat dit een begin en eind datum van de promotie is, waarbij de datums eindigend op .1 de startdatums zijn. 
Door deze om te zetten naar datums kunnen we dit later gebruiken om te analyseren hoeveel er gebruik gemaakt is van de promoties in de periode.
Eventueel vergelijkingen maken tussen periodes met en zonder promotie en de winstgevendheid hiervan.  
Het aantal promotionkeys komt overeen met het aantal in de sales set. Promotionlabel lijkt altijd gelijk te zijn aan de key, onduidelijk waarom.  
Er zitten ook waarden in de dataset met betrekking tot 2014, deze kunnen niet gerelateerd worden aan de salesdata gezien deze loopt tot 31 december 2023.
Promotion names lijken grotendeels per jaar overeen te komen. Effectiviteit over de afgelopen 3 jaar zou dus bekeken kunnen worden. 
Eventueel waarde "European Back-to-Scholl Promotion" opschonen naar School.

Key 1 lijkt een doorlopende promotie te zijn om verkopen met geen korting vast te leggen.
Is er correlatie tussen promotion en discountamount? 
De volgende stap is om de samenhang met de salesdata te analyseren. Er is hier een aansluiting op de SalesKey.



In [64]:
display(promotions_df["PromotionKey"].isin(datesales["PromotionKey"]).value_counts())
display(datesales["PromotionKey"].isin(promotions_df["PromotionKey"]).value_counts())
#Controleren welke promotionkeys in de salesset zitten en vice versa.


PromotionKey
True    28
Name: count, dtype: int64

PromotionKey
True    2282482
Name: count, dtype: int64

In [65]:
datesales.loc[datesales["PromotionKey"] == 1, "DiscountAmount"].sum()
#Controleren of alle gegevens met promotionkey 1 geen discoutn hebben. 

np.float64(0.0)

In [66]:
promo2 = datesales.loc[datesales["PromotionKey"]==2].sort_index()
#Alle waarden voor promo 2 opzoeken en sorteren op index en in aparte dataframe zetten.

In [67]:
#Controleren discountamount op basis van promotionkey 2.
#Verwachting DiscountPct is 5% van aanschafwaarde
promo2["SalesBeforeDiscount"] = promo2["UnitPrice"]*promo2["SalesQuantity"]
promo2["DiscountPct"] = promo2["DiscountAmount"]/promo2["SalesBeforeDiscount"]*100
#Geen directe aansluiting tussen percentage en discountamount.
#Vermoeden UnitPrice*DiscountQuantity* DiscountPercent = DiscountAmount
promo2["DiscountPercent"] = promo2["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
#Discountpercentage koppelen aan waarde in promo dataframe. Zoek promotion key in promo en match deze op promotion dataset, voegt verolgens de discountpercent toe als kolom.
promo2["DiscountControle"] = promo2["UnitPrice"]*promo2["DiscountQuantity"]*promo2["DiscountPercent"]
#Discount herrekenen ter controle
promo2a = promo2.loc[(promo2["DiscountControle"].round(2) != promo2["DiscountAmount"].round(2)),["DiscountAmount","DiscountControle"]]
promo2a["ControleDiscountAmount"] = promo2a["DiscountAmount"] - promo2a["DiscountControle"]
promo2a.head(50)
promo2a[(promo2a["ControleDiscountAmount"]<-0.001)|(promo2a["ControleDiscountAmount"]>0.001)]
#Controleren afwijkingen, lijkt alleen op afrondingen terecht te komen.

,DiscountAmount,DiscountControle,ControleDiscountAmount
DateKey,,,


Op basis van PromotionKey 2 is het verband tussen de PromotionKey het discountpercentage in de promotions dataset en de DiscountAmount in de sales data set geanalyseerd.  
De UnitPrice * DiscountQuantity * DiscountPercent is gelijk aan DiscountAmount voor deze promocode.  
Op basis hiervan lijken er specifieke voorwaarden aan promotieacties te zitten over de hoeveelheden waarop korting wordt geleverd.  
De volgende actie is om deze controle voor alle promotioncodes uit te voeren in de salesdataset.

In [68]:
promocontrole = datesales.copy()
#Kopieeren van salesdataset
promocontrole["DiscountPercent"] = promocontrole["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
#Mappen van discountpercentage
promocontrole["DiscountControle"] = promocontrole["UnitPrice"]*promocontrole["DiscountQuantity"]*promocontrole["DiscountPercent"]
#Berekeken discountamount
promocontrole["ControleDiscountAmount"] = promocontrole["DiscountAmount"]-promocontrole["DiscountControle"]
display(promocontrole[promocontrole["ControleDiscountAmount"] != 0])
#Controleren waarden niet gelijk aan 0
display(promocontrole[(promocontrole["ControleDiscountAmount"]<-0.001)|(promocontrole["ControleDiscountAmount"]>0.001)])
#Controleren afrondingsverschillen.

,SalesKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,
2011-12-15,686,1,226,1260,10,18.86,36.99,4,0,0.00,1,7.40,75.44,140.56,0.20,7.40,-0.00
2011-10-11,3394,1,250,1139,10,150.84,328.00,4,0,0.00,1,65.60,603.36,"1,246.40",0.20,65.60,-0.00
2011-12-01,4301,1,263,70,10,22.05,47.95,4,0,0.00,1,9.59,88.20,182.21,0.20,9.59,-0.00
2011-11-06,4678,1,226,1272,10,3.54,6.95,4,0,0.00,1,1.39,14.16,26.41,0.20,1.39,-0.00
2011-10-17,8615,1,304,1423,10,86.91,189.00,4,0,0.00,1,37.80,347.64,718.20,0.20,37.80,-0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2011-01-15,3177099,3,200,536,2,50.47,99.00,32,3,297.00,9,44.55,"1,615.04","3,123.45",0.05,44.55,-0.00
2011-11-25,3196138,2,199,128,4,73.11,143.40,18,3,430.20,3,86.04,"1,315.98","2,495.16",0.20,86.04,-0.00
2013-03-09,3279273,4,309,341,26,444.69,967.00,60,3,"2,901.00",14,947.66,"25,347.33","57,072.34",0.07,947.66,-0.00


,SalesKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,


Nu testen we dit voor de gehele dataset.

In [69]:
datesales["DiscountPercent"] = datesales["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
datesales["DiscountControle"] = datesales["UnitPrice"]*datesales["DiscountQuantity"]*datesales["DiscountPercent"]
datesales["ControleDiscountAmount"] = datesales["DiscountAmount"]-datesales["DiscountControle"]
#Herhalen stappen voor controle

display(datesales[datesales["ControleDiscountAmount"] != 0])
#Dataframe weergeven waar controle waarde niet gelijk is aan 0.
display(datesales[(datesales["ControleDiscountAmount"]<-0.001)|(datesales["ControleDiscountAmount"]>0.001)])
#Dataframe met waardes met een afwijking groter dan een duizendste achter de decimaal.

,SalesKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,
2011-12-15,686,1,226,1260,10,18.86,36.99,4,0,0.00,1,7.40,75.44,140.56,0.20,7.40,-0.00
2011-10-11,3394,1,250,1139,10,150.84,328.00,4,0,0.00,1,65.60,603.36,"1,246.40",0.20,65.60,-0.00
2011-12-01,4301,1,263,70,10,22.05,47.95,4,0,0.00,1,9.59,88.20,182.21,0.20,9.59,-0.00
2011-11-06,4678,1,226,1272,10,3.54,6.95,4,0,0.00,1,1.39,14.16,26.41,0.20,1.39,-0.00
2011-10-17,8615,1,304,1423,10,86.91,189.00,4,0,0.00,1,37.80,347.64,718.20,0.20,37.80,-0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2011-01-15,3177099,3,200,536,2,50.47,99.00,32,3,297.00,9,44.55,"1,615.04","3,123.45",0.05,44.55,-0.00
2011-11-25,3196138,2,199,128,4,73.11,143.40,18,3,430.20,3,86.04,"1,315.98","2,495.16",0.20,86.04,-0.00
2013-03-09,3279273,4,309,341,26,444.69,967.00,60,3,"2,901.00",14,947.66,"25,347.33","57,072.34",0.07,947.66,-0.00


,SalesKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,


Bovenstaande controle bevestigt het vermoeden dat UnitPrice * DiscountQuantity * DiscountPercent gelijk is aan DiscountAmount.
Er lijkt dus een verband te zijn tussen promotion en de hoeveelheid items. 
Promotions kunnen bijvoorbeeld zijn vanaf een bepaalde hoeveelheid of bij een bepaalde hoeveelheid een product gratis. Het is niet helemaal duidelijk voor iedere promotie wat deze verbanden zijn en of deze relevant zijn.

In [70]:
datesales.loc[(datesales["PromotionKey"]==2)&(datesales["DiscountAmount"]>0)].sort_index().head(50)
#Kijken naar eerste rijen met DiscountQuantity en PromotionKey 2.
datesales.loc[(datesales["PromotionKey"]==2)&(datesales["DiscountAmount"]>0)].sort_index().tail(50)
#Laate 50 waarden


,SalesKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,
2011-03-31,2000489,1,31,1197,2,220.25,432.00,9,0,0.00,2,43.20,"1,982.25","3,844.80",0.05,43.20,0.00
2011-03-31,2006706,1,38,1474,2,95.65,208.00,9,0,0.00,2,20.80,860.85,"1,851.20",0.05,20.80,0.00
2011-03-31,43013,1,116,563,2,760.38,"2,295.00",9,0,0.00,4,459.00,"6,843.42","20,196.00",0.05,459.00,0.00
2011-03-31,2126451,1,38,719,2,75.54,228.00,9,0,0.00,2,22.80,679.86,"2,029.20",0.05,22.80,0.00
2011-03-31,32220,1,138,411,2,321.44,699.00,9,0,0.00,4,139.80,"2,892.96","6,151.20",0.05,139.80,0.00
2011-03-31,29377,1,64,1412,2,132.90,289.00,9,0,0.00,4,57.80,"1,196.10","2,543.20",0.05,57.80,-0.00
2011-03-31,9441,1,7,1336,2,20.15,43.81,9,0,0.00,4,8.76,181.35,385.53,0.05,8.76,0.00
2011-03-31,1432873,2,199,818,2,5.29,11.50,9,0,0.00,3,1.73,47.61,101.78,0.05,1.73,0.00
2011-03-31,2069291,1,187,652,2,55.64,121.00,9,0,0.00,2,12.10,500.76,"1,076.90",0.05,12.10,-0.00


In [71]:
datesales.loc[datesales["PromotionKey"]==2].groupby("SalesQuantity")["DiscountQuantity"].describe()
#Kijken naar ratio's verkoop hoeveelheid afgezet tegen hoeveelheid discounts.

,count,mean,std,min,25%,50%,75%,max
SalesQuantity,,,,,,,,
4,"40,938.00",2.00,1.41,0.00,1.00,2.00,3.00,4.00
6,101.00,2.19,1.44,0.00,1.00,2.00,3.00,4.00
8,"4,480.00",3.30,2.05,0.00,2.00,3.00,5.00,8.00
9,"68,654.00",2.00,1.41,0.00,1.00,2.00,3.00,4.00
12,"3,031.00",2.85,2.29,0.00,1.00,3.00,4.00,12.00
16,931.00,4.55,2.62,0.00,3.00,4.00,6.00,14.00
18,"7,286.00",3.31,2.06,0.00,2.00,3.00,5.00,8.00
20,29.00,10.00,2.96,3.00,8.00,10.00,12.00,15.00
24,563.00,4.81,2.69,0.00,3.00,5.00,7.00,12.00


Voor nu lijkt er verder geen logisch verband te zijn tussen promotion acties en wanneer deze wel of niet wordt gegeven en op welke hoeveelheid producten.

## Voorlopige Conclusie
De promotieset bevat data over 28 promoties die open vanaf 1 januari 2011 tot en met 31 maart 2014.
De Promotion met Key 1 lijkt de standaardwaarde te zijn voor verkopen zonder enige korting.
Het verband met de salesset is het feit dat UnitPrice * DiscountQuantity * DiscountPercent gelijk is aan de DiscountAmount.

Er is geen duidelijk verband te vinden tussen de verschillende promotieacties en hoeveelheden. De verwachting zou zijn dat kortingen gelden vanaf een aantal items of per aantal items worden toegepast. Dit verband is echter niet te herleiden.

**To-do**:
- De set heeft 2 datumwaarden. De eerste heeft betrekking op de einddatum de tweede op de begindatum van de promotion. Deze dienen nog duidelijk benoemd te worden. 
- Er zit een fout in de omschrijving van de Europese back-to-scholl actie. Deze nog aanpassen naar "European Back-to-School Promotion".

In [72]:
promotions_df["EndMonth"] = pd.to_datetime(promotions_df["Month"],format="%B").dt.month.astype("int64")
#Met behulp van datetimeseries de maandnamen omzetten naar integers om later de datum mee te berekenen.
kolom = promotions_df.pop("EndMonth")
promotions_df.insert(3,"EndMonth", kolom)
#Kolom verplaatsen naar begin met rest van data.
promotions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   DiscountPercent  28 non-null     float64
 1   Year             28 non-null     int64  
 2   Quarter          28 non-null     str    
 3   EndMonth         28 non-null     int64  
 4   Month            28 non-null     str    
 5   Day              28 non-null     int64  
 6   PromotionKey     28 non-null     int64  
 7   PromotionLabel   28 non-null     int64  
 8   PromotionName    28 non-null     str    
 9   Year.1           28 non-null     int64  
 10  Quarter.1        28 non-null     str    
 11  Month.1          28 non-null     str    
 12  Day.1            28 non-null     int64  
dtypes: float64(1), int64(7), str(5)
memory usage: 4.3 KB


In [73]:
promotions_df["PromotionEndDate"] = pd.to_datetime({"year":promotions_df["Year"],"month":promotions_df["EndMonth"],"day":promotions_df["Day"]})
#Op basis van jaar, maand en, dag gegevens einddatum als datumwaarde aanmaken in de set.

In [74]:
promotions_df["StartMonth"] = pd.to_datetime(promotions_df["Month.1"],format="%B").dt.month.astype("int64")
#Startmaand converteren naar integer
promotions_df["PromotionStartDate"] = pd.to_datetime({"year":promotions_df["Year.1"],"month":promotions_df["StartMonth"],"day":promotions_df["Day.1"]})
#Op basis van jaar, maand en, dag gegevens einddatum als datumwaarde aanmaken in de set.


In [75]:
promotions_df = promotions_df.rename(columns={"Year": "EndYear",
                              "Month": "EndMonthstr",
                              "Quarter": "EndQuarter",
                              "Day": "EndDay",
                              "Year.1":"StartYear",
                              "Quarter.1":"StartQuarter",
                              "Month.1":"StartMonthstr",
                              "Day.1":"StartDay"})
#Kolomnamen hernoemen naar duidelijk benamingen die eind en begin aangeven.

In [76]:
promotions_df.loc[promotions_df["PromotionName"]== "European Back-to-Scholl Promotion","PromotionName"] = "European Back-to-School Promotion"
#Corrigeren foutieve schrijfwize European Back-to-School Promotion


In [77]:
kolom = promotions_df.pop("StartMonth")
promotions_df.insert(11,"StartMonth", kolom)
#Verplaatsen StartMonth

In [78]:
promotions_df.to_parquet(r"PQ/Contoso_promotions.parquet")

De eind-, en startdatumwaarde zijn duidelijk benoemd binnen de dataset.
Deze zijn ook geconverteerd tot bruikbare datevalues. Deze maken ook duidelijk dat de laatste promotion met 0% op het moment tot en met  eind 2014 loopt. 
De datum set zal hier op aangepast worden. 
Verder is de foutieve schrijfwijze van de European Back-to-School promotion aangepast.

# 7 ***Analyse Product (Sub)Category***

In [79]:
productcat_path= Path(r"Data\Contoso_Product_Category.csv")
#Inlezen product categories.csv
productcat_df = pd.read_csv(productcat_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.

In [80]:
init_analyse(productcat_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   ProductCategory     8 non-null      str  
 1   ProductCategoryKey  8 non-null      int64
dtypes: int64(1), str(1)
memory usage: 377.0 bytes

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


ProductCategory       0
ProductCategoryKey    0
dtype: int64


Aantal unieke waarden:


ProductCategory       8
ProductCategoryKey    8
dtype: int64


Eerste 20 rijen:


,ProductCategory,ProductCategoryKey
0,Audio,1
1,Cameras and camcorders,4
2,Cell phones,5
3,Computers,3
4,Games and Toys,7
5,Home Appliances,8
6,"Music, Movies and Audio Books",6
7,TV and Video,2



Laatste 20 rijen:


,ProductCategory,ProductCategoryKey
0,Audio,1
1,Cameras and camcorders,4
2,Cell phones,5
3,Computers,3
4,Games and Toys,7
5,Home Appliances,8
6,"Music, Movies and Audio Books",6
7,TV and Video,2


Dataset lijkt een subcategory te zijn van een andere dataset. 
Vermoedelijk de Product Sub Category dataset.
Verbinding lijkt te zijn ProductKey -> ProductSubCategroy -> ProductCategory
Eerst product Sub Category bekijken.

In [81]:
productsubcat_path= Path(r"Data\Contoso_ProductSubCategory.csv")
#Inlezen product subcategories.csv
productsubcat_df = pd.read_csv(productsubcat_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.

In [82]:
init_analyse(productsubcat_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   ProductCategoryKey     44 non-null     int64
 1   ProductSubcategory     44 non-null     str  
 2   ProductSubcategoryKey  44 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.8 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


ProductCategoryKey       0
ProductSubcategory       0
ProductSubcategoryKey    0
dtype: int64


Aantal unieke waarden:


ProductCategoryKey        8
ProductSubcategory       44
ProductSubcategoryKey    44
dtype: int64


Eerste 20 rijen:


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
0,1,Audio Accessories,8
1,1,Bluetooth Headphones,6
2,1,Headphones,5
3,1,MP4&MP3,1
4,1,Radio,3
5,1,Recorder,2
6,1,Recording Pen,4
7,1,Speakers,7
8,2,Car Video,13
9,2,Home Theater System,11



Laatste 20 rijen:


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
24,4,Digital SLR Cameras,24
25,4,Film Cameras,25
26,5,Cell phones Accessories,33
27,5,Home & Office Phones,29
28,5,Smart phones & PDAs,32
29,5,Touch Screen Phones,31
30,6,Audio Books,36
31,6,Movie DVD,35
32,6,Music CD,34
33,7,Boxed Games,38


Er lijken 44 unieke Subcategorieën te zijn, verbonden aan 1 van de 8 hoofdcategorieën.
Volgende stap is om te kijken hoeveel van de subcategorykeys zich in de productlijst bevinden.

De header ProductSubcategory is niet in consistente camelcase

In [83]:
masker = productsubcat_df["ProductSubcategoryKey"].isin(product_df["ProductSubcategoryKey"])
#filter die controleert welke keys wel of niet in de product dataset zitten.
print(f"{productsubcat_df.loc[masker,["ProductSubcategoryKey"]].nunique().sum()} Productsubcategorieën in product dataframe")
display(productsubcat_df[masker])
#Geeft alle waarden weer die wel in de dataset zitten.
print(f"{productsubcat_df.loc[~masker,["ProductSubcategoryKey"]].nunique().sum()} Productsubcategorieën niet in product dataframe")
display(productsubcat_df[~masker])

22 Productsubcategorieën in product dataframe


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
1,1,Bluetooth Headphones,6
3,1,MP4&MP3,1
6,1,Recording Pen,4
8,2,Car Video,13
9,2,Home Theater System,11
10,2,Televisions,9
12,2,VCD & DVD,10
14,3,Computers Accessories,22
15,3,Desktops,17
16,3,Laptops,15


22 Productsubcategorieën niet in product dataframe


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
0,1,Audio Accessories,8
2,1,Headphones,5
4,1,Radio,3
5,1,Recorder,2
7,1,Speakers,7
11,2,TV & Video Accessories,14
13,3,Computer Setup & Service,21
18,3,Netbooks,16
25,4,Film Cameras,25
30,6,Audio Books,36


Dataset lijkt met namen hardware en multimedia apparaten te betreffen.  De 22 waarden sluiten ook aan met de unique count van de productset.

De productgroep die niet in de productcategory voorkomt heet met name berekking op witgoed, media en audio.

In [84]:
productsubcat_df["ProductCategoryKey"].isin(productcat_df["ProductCategoryKey"])
productcat_df["ProductCategoryKey"].isin(productsubcat_df["ProductCategoryKey"])
#Controleren of alle keys in de productcategoryKey zitten.

0    True
1    True
2    True
3    True
4    True
5    True
6    True
7    True
Name: ProductCategoryKey, dtype: bool

In [85]:
print("Categorieën in product data set:")
display(productcat_df[(productcat_df["ProductCategoryKey"]).isin(productsubcat_df.loc[masker,"ProductCategoryKey"])])
#Controleren welke productcategorieën voorkomen in products
print("Categorieën niet in product data set:")
productcat_df[~(productcat_df["ProductCategoryKey"]).isin(productsubcat_df.loc[masker,"ProductCategoryKey"])]
#Controleren welke productcategorieën niet voorkomen in products

Categorieën in product data set:


,ProductCategory,ProductCategoryKey
0,Audio,1
1,Cameras and camcorders,4
2,Cell phones,5
3,Computers,3
6,"Music, Movies and Audio Books",6
7,TV and Video,2


Categorieën niet in product data set:


,ProductCategory,ProductCategoryKey
4,Games and Toys,7
5,Home Appliances,8


Conform de hypothese zijn witgoed en media uitgesloten van de set.

## Voorlopige conclusie

De dataset lijkt 22 van de 44 subcategorieën te gebruiken.
Daarnaast zijn de categoriën met ProductCategoryKey 7 en 8 niet van toepassing op de sales dataset.

**Actiepunten**:
- Eventueel ongebruikte categorieën laten vallen, gezien de hoeveelheid lijkt de impact hiervan niet al te groot op performance.

In [86]:
productcat_df.to_parquet(r"PQ/Contoso_productcat.parquet")
productsubcat_df.to_parquet(r"PQ/Contoso_productsubcat.parquet")

# 9 ***Analyse Channel data***

In [87]:
channel_path= Path(r"Data\Contoso_Channel.csv")
#Inlezen product channel.csv
channel_df = pd.read_csv(channel_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.

In [88]:
channel_df.to_parquet(r"PQ/Contoso_channel.parquet")

Er zijn 4 channels, deze corresponderen met de channelkeys in de sales set.